# SVA Inference: Base Qwen + 3 Adapters

Runs batch inference on the test set (578 modules) using vLLM for:
1. Base Qwen2.5-Coder-7B-Instruct (no adapter)
2. Adapter: `all`
3. Adapter: `syntax_pass`
4. Adapter: `verified`

Outputs saved to Google Drive in JasperGold-ready folder structure.

## 0. Install Dependencies

In [ ]:
!pip install -q vllm datasets huggingface_hub

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.9/87.9 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 433.2/433.2 MB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 192.6/192.6 kB 21.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.8/7.8 MB 192.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.0/111.0 kB 14.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.4/45.4 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.9/3.9 MB 168.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 112.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 469.4/469.4 kB 53.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.2/114.2 kB 14.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.

## 1. Configuration

In [ ]:
import os
import json
from pathlib import Path

# Model
BASE_MODEL = "Qwen/Qwen2.5-Coder-7B-Instruct"

# Dataset
DATASET_REPO = "aarushgoradia/malik25_26"
TEST_CONFIG = "all"  # use the full test set

# Adapters on HuggingFace
ADAPTERS = {
    "adapter_all": "aarushgoradia/sva-qlora-all",
    "adapter_syntax_pass": "aarushgoradia/sva-qlora-syntax_pass",
    "adapter_verified": "aarushgoradia/sva-qlora-verified",
}

# Generation parameters
MAX_NEW_TOKENS = 2048
TEMPERATURE = 0.1
TOP_P = 0.95
MAX_MODEL_LEN = 8192
LORA_RANK = 64

# Output
OUTPUT_BASE = Path("/content/drive/MyDrive/sva_qlora/inference_outputs")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import torch
print(f"GPU: {torch.cuda.get_device_name(0)}")

Mounted at /content/drive
GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition


In [ ]:
from huggingface_hub import notebook_login
notebook_login()

## 2. Load Test Set & Build Prompts

In [ ]:
from datasets import load_dataset

test_dataset = load_dataset(DATASET_REPO, TEST_CONFIG, split="test")
print(f"Test set: {len(test_dataset)} modules")

README.md: 0.00B [00:00, ?B/s]

all/train-00000-of-00001.parquet:   0%|          | 0.00/3.56M [00:00<?, ?B/s]

all/validation-00000-of-00001.parquet:   0%|          | 0.00/472k [00:00<?, ?B/s]

all/test-00000-of-00001.parquet:   0%|          | 0.00/462k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/4619 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/577 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/578 [00:00<?, ? examples/s]

Test set: 578 modules


In [ ]:
SYSTEM_PROMPT = """\
You are an expert SystemVerilog verification engineer writing SVA for Jasper \
formal verification.
OUTPUT REQUIREMENTS:
1. Output a single, complete .sv file that compiles in Jasper without modification.
2. The file must be a module that takes the DUT's ports as inputs and contains \
   SVA properties bound to those signals. You can use internal signals \
   if they are present in the RTL, but do NOT invent new ones.
3. Every property must use a clocked event (@(posedge clk) or the appropriate \
   clock from the RTL). NEVER use combinational or level-sensitive events in \
   property statements — Jasper rejects these. For modules with combinational \
   logic, still clock your assertions to the appropriate clock edge.
4. Use `disable iff` with the correct reset polarity as shown in the RTL.
5. Use descriptive labels for every assertion (e.g., `check_grant_mutex`, not `a1`).
6. Add a brief comment above each assertion explaining what it checks.
7. Only assert behaviors that the RTL actually implements. Do not invent \
   signals, states, or protocols that are not present in the code.
8. Focus on QUALITY and CORRECTNESS — 10 correct, meaningful assertions are \
   worth more than 30 trivial or speculative ones.
9. Do NOT wrap output in markdown code fences or add explanation outside the code.
10. Keep comments minimal — one short line per assertion. Do NOT include large \
   comment blocks, file headers, or explanations of your approach.
REFERENCE EXAMPLE — this is the style and quality level to target:
```systemverilog
module manual (
    input logic CLK,
    input logic RESETn,
    input logic QREQn,
    input logic QACCEPTn,
    input logic QDENY,
    input logic QACTIVE
);
    ///// Handshake rules /////
    // QREQn can only transition from HIGH to LOW when QACCEPTn is HIGH and QDENY is LOW.
    handshake_1: assume property (
        @(posedge CLK) disable iff (!RESETn) $fell(QREQn) |-> (QACCEPTn == 1'b1) && (QDENY == 1'b0)
    );
    // QACCEPTn can only transition from HIGH to LOW when QREQn is LOW and QDENY is LOW.
    handshake_3: assert property (
        @(posedge CLK) disable iff (!RESETn) $fell(QACCEPTn) |-> (QREQn == 1'b0) && (QDENY == 1'b0)
    );
    // QDENY can only transition from LOW to HIGH when QREQn is LOW and QACCEPTn is HIGH.
    handshake_6: assert property (
        @(posedge CLK) disable iff (!RESETn) $rose(QDENY) |-> (QREQn == 1'b0) && (QACCEPTn == 1'b1)
    );
    ///// Device reset /////
    // At reset assertion, a device must drive both QACCEPTn and QDENY LOW.
    reset: assert property (
        @(posedge CLK) !RESETn |-> (QACCEPTn == 1'b0) && (QDENY == 1'b0)
    );
endmodule
```
Note the pattern: module wrapper with DUT ports as inputs, descriptive labels, \
comments explaining intent, proper clocking and reset disable on every property, \
and appropriate use of assume vs assert."""


def make_user_prompt(rtl_code: str) -> str:
    return f"""Analyze the following RTL module carefully. Identify:
- The clock(s) and reset signal(s), including reset polarity
- Whether the logic is sequential, combinational, or mixed
- The key signals, interfaces, and functional behaviors
Then generate a complete .sv assertion module following the style shown in \
your reference example. Only write assertions for behaviors that are actually \
present in this RTL — do not guess or assume functionality that isn't there. \
For combinational logic, still use clocked assertions (@(posedge clk)).
RTL module:
```verilog
{rtl_code}
```"""

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)

# Build prompts for all test samples
prompts = []
sample_ids = []
sample_rtls = []

for example in test_dataset:
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": make_user_prompt(example["rtl"])},
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    prompts.append(text)
    sample_ids.append(example["id"])
    sample_rtls.append(example["rtl"])

print(f"Built {len(prompts)} prompts")
print(f"Sample prompt length (tokens, approx): {len(tokenizer.encode(prompts[0]))}")

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Built 578 prompts
Sample prompt length (tokens, approx): 1019


## 3. Helper: Save Results

In [ ]:
def save_results(condition_name: str, outputs, sample_ids, sample_rtls):
    """Save generated SVA files in JasperGold-ready folder structure."""
    condition_dir = OUTPUT_BASE / condition_name

    saved = 0
    for i, output in enumerate(outputs):
        generated_sva = output.outputs[0].text
        sid = sample_ids[i]

        # Create sample directory
        sample_dir = condition_dir / sid
        sample_dir.mkdir(parents=True, exist_ok=True)

        # Save the generated SVA
        sva_path = sample_dir / "sva.sv"
        with open(sva_path, "w", encoding="utf-8") as f:
            f.write(generated_sva)

        # Save metadata
        meta = {
            "id": sid,
            "condition": condition_name,
            "model": BASE_MODEL,
            "temperature": TEMPERATURE,
            "max_new_tokens": MAX_NEW_TOKENS,
            "prompt_tokens": len(output.prompt_token_ids),
            "completion_tokens": len(output.outputs[0].token_ids),
        }
        meta_path = sample_dir / "metadata.json"
        with open(meta_path, "w", encoding="utf-8") as f:
            json.dump(meta, f, indent=2)

        saved += 1

    print(f"[{condition_name}] Saved {saved} SVA files to {condition_dir}")
    return condition_dir

## 4. Download Adapters

In [ ]:
from huggingface_hub import snapshot_download

adapter_paths = {}
for name, repo in ADAPTERS.items():
    local_path = f"/content/adapters/{name}"
    print(f"Downloading {repo}...")
    snapshot_download(repo_id=repo, local_dir=local_path)
    adapter_paths[name] = local_path
    print(f"  Saved to {local_path}")

print(f"\nAll adapters downloaded: {list(adapter_paths.keys())}")

Fetching 15 files:   0%|          | 0/15 [00:00<?, ?it/s]

adapter_model.safetensors:   0%|          | 0.00/646M [00:00<?, ?B/s]

final_adapter/adapter_model.safetensors:   0%|          | 0.00/646M [00:00<?, ?B/s]

.gitattributes: 0.00B [00:00, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

final_adapter/training_args.bin:   0%|          | 0.00/5.71k [00:00<?, ?B/s]

training_args.bin:   0%|          | 0.00/5.71k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/664 [00:00<?, ?B/s]

  Saved to /content/adapters/adapter_verified

All adapters downloaded: ['adapter_all', 'adapter_syntax_pass', 'adapter_verified']


## 5. Run Inference — Base Qwen (No Adapter)

Load vLLM with LoRA enabled (needed for adapter runs later), but first run without any adapter.

In [ ]:
from vllm import LLM, SamplingParams

llm = LLM(
    model=BASE_MODEL,
    enable_lora=True,
    max_lora_rank=LORA_RANK,
    max_model_len=MAX_MODEL_LEN,
    trust_remote_code=True,
    dtype="bfloat16",
    gpu_memory_utilization=0.90,
)

sampling_params = SamplingParams(
    temperature=TEMPERATURE,
    top_p=TOP_P,
    max_tokens=MAX_NEW_TOKENS,
)

print("vLLM loaded. Running base Qwen inference...")

INFO 03-31 14:55:48 [utils.py:233] non-default args: {'trust_remote_code': True, 'dtype': 'bfloat16', 'max_model_len': 8192, 'disable_log_stats': True, 'enable_lora': True, 'max_lora_rank': 64, 'model': 'Qwen/Qwen2.5-Coder-7B-Instruct'}
INFO 03-31 14:56:02 [model.py:533] Resolved architecture: Qwen2ForCausalLM
INFO 03-31 14:56:02 [model.py:1582] Using max model len 8192
INFO 03-31 14:56:02 [scheduler.py:231] Chunked prefill is enabled with max_num_batched_tokens=16384.
INFO 03-31 14:56:02 [vllm.py:775] Asynchronous scheduling is enabled.


generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

WARNING 03-31 14:56:03 [system_utils.py:152] We must use the `spawn` multiprocessing start method. Overriding VLLM_WORKER_MULTIPROC_METHOD to 'spawn'. See https://docs.vllm.ai/en/latest/usage/troubleshooting.html#python-multiprocessing for more information. Reasons: CUDA is initialized
INFO 03-31 14:58:24 [llm.py:391] Supported tasks: ['generate']
vLLM loaded. Running base Qwen inference...


In [ ]:
# Base Qwen — no LoRA adapter
base_outputs = llm.generate(prompts, sampling_params)
save_results("base_qwen", base_outputs, sample_ids, sample_rtls)
print("Base Qwen inference complete!")

Rendering prompts:   0%|          | 0/578 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/578 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

[base_qwen] Saved 578 SVA files to /content/drive/MyDrive/sva_qlora/inference_outputs/base_qwen
Base Qwen inference complete!


## 6. Run Inference — All 3 Adapters

In [ ]:
from vllm.lora.request import LoRARequest

for i, (condition_name, adapter_path) in enumerate(adapter_paths.items(), start=1):
    print(f"\n{'='*60}")
    print(f"Running inference: {condition_name} ({i}/3)")
    print(f"Adapter: {adapter_path}")
    print(f"{'='*60}")

    lora_request = LoRARequest(
        lora_name=condition_name,
        lora_int_id=i,
        lora_local_path=adapter_path,
    )

    outputs = llm.generate(prompts, sampling_params, lora_request=lora_request)
    save_results(condition_name, outputs, sample_ids, sample_rtls)

print(f"\n{'='*60}")
print("ALL INFERENCE COMPLETE!")
print(f"{'='*60}")

## 7. Copy ChatGPT Baseline

The ChatGPT baseline is the original `sva.sv` from the test set. Save it in the same structure for easy comparison.

In [ ]:
chatgpt_dir = OUTPUT_BASE / "chatgpt_baseline"

for example in test_dataset:
    sid = example["id"]
    sample_dir = chatgpt_dir / sid
    sample_dir.mkdir(parents=True, exist_ok=True)

    # Save original SVA (ChatGPT-generated)
    with open(sample_dir / "sva.sv", "w", encoding="utf-8") as f:
        f.write(example["sva"])

    # Save metadata
    meta = {"id": sid, "condition": "chatgpt_baseline", "model": "gpt-4o"}
    with open(sample_dir / "metadata.json", "w", encoding="utf-8") as f:
        json.dump(meta, f, indent=2)

print(f"Saved {len(test_dataset)} ChatGPT baseline files to {chatgpt_dir}")

## 8. Quick Summary

In [ ]:
print("\nInference Output Summary")
print("=" * 50)
for condition in ["base_qwen", "adapter_all", "adapter_syntax_pass", "adapter_verified", "chatgpt_baseline"]:
    cdir = OUTPUT_BASE / condition
    if cdir.exists():
        count = len([d for d in cdir.iterdir() if d.is_dir()])
        print(f"{condition:<25} {count} modules")
    else:
        print(f"{condition:<25} NOT FOUND")

print(f"\nAll outputs saved to: {OUTPUT_BASE}")
print("\nNext step: Run JasperGold on each condition folder.")

## 9. Spot Check

Compare outputs from all conditions for one sample.

In [ ]:
SPOT_CHECK_ID = sample_ids[0]  # change to any test ID

for condition in ["base_qwen", "adapter_all", "adapter_syntax_pass", "adapter_verified", "chatgpt_baseline"]:
    sva_path = OUTPUT_BASE / condition / SPOT_CHECK_ID / "sva.sv"
    if sva_path.exists():
        sva = sva_path.read_text()
        num_assertions = sva.count("assert property") + sva.count("assume property")
        print(f"\n{'='*60}")
        print(f"{condition} — {num_assertions} assertions")
        print(f"{'='*60}")
        print(sva[:800])
        if len(sva) > 800:
            print("...")